# Example script for Hackathon

Within each cycle of active learning, you can:

1. Collect training data (original training data + your query data).

2. Train a prediction model to predict the DMS_score for each mutant (e.g., M0A).

3. Use the trained model to predict the score for all mutant in the test set.

4. Select query mutants for next round based on certain criteria. You may want to make sure you don't query the same mutant twice as you only have a limited chances of making queries in total.

In [219]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Dataset
import random
from copy import deepcopy
import pandas as pd
from scipy.stats import spearmanr
import argparse
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor

## 1. collect training data

Upload `sequence.fasta`, `train.csv`, and `test.csv` to the current runtime:

1. click the folder icon on the left

2. click the upload icon and upload the files to the current directory

In [220]:
with open('sequence.fasta', 'r') as f:
  data = f.readlines()

sequence_wt = data[1].strip()
sequence_wt

'MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLREKMRRRLESGDKWFSLEFFPPRTAEGAVNLISRFDRMAAGGPLYIDVTWHPAGDPGSDKETSSMMIASTAVNYCGLETILHMTCCRQRLEEITGHLHKAKQLGLKNIMALRGDPIGDQWEEEEGGFNYAVDLVKHIRSEFGDYFDICVAGYPKGHPEAGSFEADLKHLKEKVSAGADFIITQLFFEADTFFRFVKACTDMGITCPIVPGIFPIQGYHSLRQLVKLSKLEVPQEIKDVIEPIKDNDAAIRNYGIELAVSLCQELLASGLVPGLHFYTLNREMATTEVLKRLGMWTEDPRRPLPWALSAHPKRREEDVRPIFWASRPKSYIYRTQEWDEFPNGRWGNSSSPAFGELKDYYLFYLKSKSPKEELLKMWGEELTSEESVFEVFVLYLSGEPNRNGHKVTCLPWNDEPLAAETSLLKEELLRVNRQGILTINSQPNINGKPSSDPIVGWGPSGGYVFQKAYLEFFTSRETAEALLQVLKKYELRVNYHLVNVKGENITNAPELQPNAVTWGIFPGREIIQPTVVDPVSFMFWKDEAFALWIERWGKLYEEESPSRTIIQYIHDNYFLVNLVDNDFPLDNCLWQVVEDTLELLNRPTQNARETEAP'

In [221]:
len(sequence_wt)

656

In [222]:
def get_mutated_sequence(mut, sequence_wt):
  wt, pos, mt = mut[0], int(mut[1:-1]), mut[-1]

  sequence = deepcopy(sequence_wt)

  return sequence[:pos]+mt+sequence[pos+1:]

In [223]:
# df_train = pd.read_csv('train.csv')
# df_train['sequence'] = df_train.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
# df_train

In [224]:
df_test = pd.read_csv('test.csv')
df_test['sequence'] = df_test.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_test

,mutant,sequence
0,V1D,MDNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,V1Y,MYNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,V1C,MCNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,V1E,MENEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...
11319,P655S,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11320,P655T,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11321,P655V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11322,P655A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [225]:
import pandas as pd

df_train_full = pd.read_csv('train.csv')




In [226]:
df_train_full

,mutant,DMS_score
0,M0Y,0.2730
1,M0W,0.2857
2,M0V,0.2153
3,M0T,0.3122
4,M0S,0.2180
...,...,...
1135,P347D,0.3876
1136,P347C,0.1837
1137,P347A,0.4611
1138,P347M,0.2412


## 2. Train a prediction model

Here, we provided a linear regression model and used one-hot encoding to encode each variant. You would need to build your own model to achieve better performances.

Hint: you can perform cross-validation on the training set to evaluate your predictor before making predictions on the test set.

In [227]:
'''hyperparameters'''

seq_length = 656
seed = 0 # seed for splitting the validation set
val_ratio = 0.2 # proportion of validation set

Adding the aa_group reasoning:

This change helped because it gives the model a simple way to understand how “big” a mutation is. Before, the model just saw two letters (like A → V) with no sense of how similar they are. By grouping amino acids into categories, we're telling the model that some mutations involve similar types while others are very different. Mutations between similar types usually do not change the protein much, while very different ones can have a bigger effect. Adding this one extra feature gives the model that intuition, which helps it make better predictions on new, unseen data without making the model more complicated.

In [228]:
import numpy as np
import torch
from torch.utils.data import Dataset

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_to_idx = {aa: i for i, aa in enumerate(amino_acids)}

# BLOSUM62 Evolutionary Matrix
blosum62_raw = {('W', 'F'): 1, ('L', 'R'): -2, ('S', 'P'): -1, ('V', 'T'): 0, ('Q', 'Q'): 5, ('N', 'A'): -2, ('Z', 'Y'): -2, ('W', 'R'): -3, ('Q', 'A'): -1, ('S', 'D'): 0, ('H', 'A'): -2, ('S', 'H'): -1, ('H', 'D'): -1, ('L', 'N'): -3, ('W', 'A'): -3, ('Y', 'M'): -1, ('G', 'R'): -2, ('Y', 'I'): -1, ('Y', 'E'): -2, ('B', 'Y'): -3, ('Y', 'A'): -2, ('V', 'D'): -3, ('B', 'S'): 0, ('Y', 'Y'): 7, ('G', 'N'): 0, ('E', 'C'): -4, ('Y', 'Q'): -1, ('Z', 'Z'): 4, ('V', 'A'): 0, ('C', 'C'): 9, ('M', 'R'): -1, ('V', 'E'): -2, ('T', 'N'): 0, ('P', 'P'): 7, ('V', 'I'): 3, ('V', 'S'): -2, ('Z', 'P'): -1, ('V', 'M'): 1, ('T', 'F'): -2, ('V', 'Q'): -2, ('K', 'K'): 5, ('P', 'D'): -1, ('I', 'H'): -3, ('I', 'D'): -3, ('T', 'R'): -1, ('P', 'L'): -3, ('K', 'G'): -2, ('M', 'N'): -2, ('P', 'H'): -2, ('F', 'Q'): -3, ('Z', 'G'): -2, ('X', 'L'): -1, ('T', 'M'): -1, ('Z', 'C'): -3, ('X', 'H'): -1, ('D', 'R'): -2, ('B', 'W'): -4, ('X', 'D'): -1, ('Z', 'T'): -1, ('F', 'A'): -2, ('Z', 'W'): -3, ('F', 'E'): -3, ('D', 'N'): 1, ('B', 'K'): 0, ('X', 'P'): -2, ('D', 'J'): -3, ('X', 'T'): 0, ('Z', 'S'): 0, ('F', 'I'): 0, ('Z', 'O'): -1, ('F', 'M'): 0, ('J', 'H'): -3, ('Z', 'K'): 1, ('J', 'D'): -3, ('F', 'Y'): 3, ('X', 'R'): -1, ('Z', 'Q'): 3, ('X', 'F'): -1, ('X', 'B'): -1, ('J', 'N'): -3, ('X', 'Z'): -1, ('X', 'Y'): -1, ('X', 'W'): -2, ('X', 'V'): -1, ('F', 'C'): -2, ('X', 'Q'): -1, ('X', 'N'): -1, ('J', 'A'): -3, ('X', 'J'): -1, ('K', 'R'): 2, ('B', 'F'): -3, ('V', 'N'): -3, ('P', 'C'): -3, ('Y', 'H'): 2, ('Y', 'D'): -3, ('Y', 'L'): -1, ('G', 'A'): 0, ('P', 'O'): -2, ('Y', 'C'): -2, ('P', 'E'): -1, ('P', 'A'): -1, ('N', 'N'): 6, ('Y', 'W'): 2, ('Y', 'S'): -2, ('Y', 'T'): -2, ('B', 'D'): 4, ('B', 'H'): 0, ('B', 'L'): -4, ('W', 'N'): -4, ('Y', 'R'): -2, ('B', 'P'): -2, ('G', 'E'): -2, ('B', 'T'): -1, ('W', 'J'): -3, ('G', 'I'): -4, ('W', 'F'): 1, ('G', 'M'): -3, ('W', 'B'): -4, ('G', 'Q'): -2, ('G', 'Y'): -3, ('G', 'U'): -3, ('G', 'W'): -2, ('Y', 'V'): -1, ('E', 'R'): 0, ('B', 'X'): -1, ('G', 'C'): -3, ('E', 'N'): 0, ('W', 'V'): -3, ('W', 'R'): -3, ('E', 'A'): -1, ('W', 'Z'): -3, ('E', 'J'): -3, ('T', 'A'): 0, ('T', 'E'): -1, ('S', 'R'): -1, ('T', 'I'): -1, ('T', 'C'): -1, ('T', 'O'): -1, ('T', 'Y'): -2, ('T', 'W'): -2, ('T', 'V'): 0, ('T', 'Q'): -1, ('B', 'E'): 1, ('B', 'A'): -2, ('B', 'I'): -3, ('S', 'N'): 1, ('S', 'J'): -2, ('B', 'M'): -3, ('T', 'K'): -1, ('B', 'Q'): 0, ('B', 'U'): -4, ('I', 'R'): -3, ('S', 'A'): 1, ('S', 'F'): -2, ('B', 'C'): -3, ('I', 'N'): -3, ('S', 'B'): 0, ('I', 'A'): -1, ('S', 'M'): -1, ('S', 'Z'): 0, ('I', 'J'): 3, ('S', 'U'): -3, ('S', 'Y'): -2, ('S', 'Q'): 0, ('I', 'C'): -1, ('I', 'W'): -3, ('I', 'U'): -3, ('I', 'Y'): -1, ('I', 'Q'): -3, ('L', 'F'): 0, ('W', 'S'): -3, ('M', 'F'): 0, ('L', 'B'): -4, ('W', 'O'): -2, ('M', 'B'): -3, ('L', 'J'): 2, ('W', 'K'): -3, ('M', 'J'): 2, ('W', 'H'): -2, ('W', 'D'): -4, ('W', 'L'): -2, ('M', 'A'): -1, ('L', 'Z'): -3, ('L', 'V'): 1, ('L', 'W'): -2, ('M', 'E'): -2, ('M', 'I'): 1, ('L', 'Q'): -2, ('M', 'C'): -1, ('W', 'P'): -4, ('M', 'U'): -2, ('W', 'T'): -2, ('M', 'Y'): -1, ('M', 'W'): -1, ('M', 'Q'): 0}
blosum62 = {}
for (k1, k2), v in blosum62_raw.items():
    blosum62[(k1, k2)] = v; blosum62[(k2, k1)] = v

hydro = {'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5, 'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5, 'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6, 'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2}
vol = {'A': 88.6, 'R': 173.4, 'N': 114.1, 'D': 111.1, 'C': 108.5, 'Q': 143.8, 'E': 138.4, 'G': 60.1, 'H': 153.2, 'I': 166.7, 'L': 166.7, 'K': 168.6, 'M': 162.9, 'F': 189.9, 'P': 112.7, 'S': 89.0, 'T': 116.1, 'W': 227.8, 'Y': 193.6, 'V': 140.0}
charge = {'A': 0, 'R': 1, 'N': 0, 'D': -1, 'C': 0, 'Q': 0, 'E': -1, 'G': 0, 'H': 0.1, 'I': 0, 'L': 0, 'K': 1, 'M': 0, 'F': 0, 'P': 0, 'S': 0, 'T': 0, 'W': 0, 'Y': 0, 'V': 0}

def encode_mutation(mut, seq_length=656):
    wt_aa, pos, mut_aa = mut[0], int(mut[1:-1]), mut[-1]


    vec = [pos / seq_length]

    # MUTANT ONE-HOT ONLY
    mut_oh = [0]*20
    if mut_aa in aa_to_idx: mut_oh[aa_to_idx[mut_aa]] = 1
    vec.extend(mut_oh)

    # Biological Differences
    vec.append(blosum62.get((wt_aa, mut_aa), 0))
    vec.append(hydro.get(mut_aa, 0) - hydro.get(wt_aa, 0))
    vec.append(vol.get(mut_aa, 0) - vol.get(wt_aa, 0))
    vec.append(charge.get(mut_aa, 0) - charge.get(wt_aa, 0))

    return np.array(vec, dtype=np.float32)

class ProteinDataset(Dataset):
    def __init__(self, df, istrain=True):
        self.encodings = np.stack([encode_mutation(m) for m in df['mutant'].values])
        self.targets = df['DMS_score'].values.astype(np.float32) if istrain else np.zeros(len(df), dtype=np.float32)

    def __len__(self): return len(self.encodings)
    def __getitem__(self, idx): return torch.tensor(self.encodings[idx]), torch.tensor(self.targets[idx])

In [229]:
train_dataset = ProteinDataset(df_train)
test_dataset = ProteinDataset(df_test, istrain=False)

# split validation set
train_dataset, val_dataset = train_test_split(train_dataset, test_size=val_ratio, random_state=seed, shuffle=True)

# TODO: revise according to your own model
train_loader = DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

XGBoost Working model

In [230]:
from xgboost import XGBRegressor
import numpy as np


X_full_train = np.stack([x[0].numpy() for x in train_dataset])
y_full_train = np.array([x[1].numpy() for x in train_dataset])

X_test = np.stack([x[0].numpy() for x in test_dataset])


model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='rank:pairwise'
)


model.fit(X_full_train, y_full_train)


final_test_preds = model.predict(X_test)

ETR Gradient Boost Model with spearman score 0.3266:

Tried Ridge, GradientBoost, HistGradientBoost, and Extra Trees

Neural Network attempt

To view the head and create predictions.csv

In [231]:
df_test['DMS_score_predicted'] = final_test_preds
df_test

,mutant,sequence,DMS_score_predicted
0,V1D,MDNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,-0.888368
1,V1Y,MYNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,-1.062650
2,V1C,MCNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,-0.333266
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,-0.163029
4,V1E,MENEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,-0.818811
...,...,...,...
11319,P655S,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,-0.000050
11320,P655T,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.440718
11321,P655V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.145206
11322,P655A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.091365


In [232]:
df_test['id'] = df_test.index
df_test[['id', 'DMS_score_predicted']].rename(columns={'DMS_score_predicted': 'DMS_score'}).to_csv('predictions.csv', index=False)
# df_test[['mutant', 'DMS_score_predicted']].to_csv('predictions.csv', index=False)

## 3. Select query for next round

In [233]:
df_test.sort_values('DMS_score_predicted', ascending=False).head(100)

,mutant,sequence,DMS_score_predicted,id
10935,V635I,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,1.701539,10935
7856,V473I,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,1.701539,7856
7087,V433I,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,1.701539,7087
7047,V430I,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,1.701539,7047
9171,V542I,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,1.701539,9171
...,...,...,...,...
6615,K408R,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,1.277770,6615
5920,K371R,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,1.277770,5920
8175,K490R,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,1.277770,8175
8923,K529R,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,1.277770,8923


In [234]:
best_mutants = df_test.sort_values('DMS_score_predicted', ascending=False).head(10)['mutant'].values
with open('top10.txt', 'w') as f:
    for mut in best_mutants:
        f.write(mut + '\n')
df_test.sort_values('DMS_score_predicted', ascending=False).head(100)

,mutant,sequence,DMS_score_predicted,id
10935,V635I,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,1.701539,10935
7856,V473I,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,1.701539,7856
7087,V433I,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,1.701539,7087
7047,V430I,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,1.701539,7047
9171,V542I,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,1.701539,9171
...,...,...,...,...
6615,K408R,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,1.277770,6615
5920,K371R,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,1.277770,5920
8175,K490R,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,1.277770,8175
8923,K529R,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,1.277770,8923


In [235]:
# Example: randomly select 100 test variants to be queried.
# Note: random selection may not be a good strategy
# TODO: select query mutants for the next round based on your own criteria

querys = np.random.choice(df_test.mutant.values, size=100, replace=False)
querys


array(['N11K', 'E3K', 'K104M', 'Q570D', 'F206V', 'M110C', 'L639G',
       'E325N', 'N536G', 'L589H', 'F515I', 'L264I', 'L479I', 'E519D',
       'V435A', 'A352W', 'V178M', 'D613N', 'E427S', 'L465G', 'F79S',
       'A174E', 'G565D', 'F229C', 'V178A', 'A83F', 'C119R', 'E15F',
       'D91Y', 'Q570L', 'D628A', 'V634K', 'L525F', 'K583P', 'S20K',
       'D91M', 'N546R', 'E299Q', 'T328W', 'R324C', 'P66S', 'A510C',
       'T114V', 'F64E', 'F256M', 'R387P', 'G260L', 'L589W', 'Q609V',
       'L75E', 'I591D', 'Y261D', 'E39W', 'D341E', 'A349V', 'V558R',
       'E423L', 'E274D', 'K269M', 'I562P', 'P412C', 'A550R', 'S8M',
       'Y598N', 'C192F', 'R50G', 'K448M', 'H538L', 'E307P', 'W338M',
       'S492D', 'R133E', 'D37Q', 'T93V', 'A460S', 'I279H', 'P551A',
       'N443V', 'T548R', 'T33S', 'A219T', 'P495S', 'L479G', 'G386H',
       'P66M', 'K355F', 'G85D', 'S22C', 'A586H', 'P564E', 'N488W',
       'H140C', 'L332D', 'T638M', 'T33G', 'E167D', 'F616S', 'K269G',
       'H538D', 'E545P'], dtype=object)

In [236]:
with open('query.txt', 'w') as f:
  for mutant in querys:
    f.write(mutant+'\n')